In [ ]:
# STEP 1: Mount Google Drive — makes trained models/data persist across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/ecommerce-customer-intelligence"
print(f"Project will live at: {PROJECT_DIR}")

In [ ]:
# STEP 2: Clone/pull the project from GitHub (edit GITHUB_REPO_URL if using a different repo).
GITHUB_REPO_URL = "https://github.com/rayyantaimoor1/Ecommerce-customer-intelligence.git"

import os

if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repo already exists — pulling latest changes...")
    os.chdir(PROJECT_DIR)
    os.system("git pull")
else:
    print(f"Cloning {GITHUB_REPO_URL} ...")
    os.system(f'git clone "{GITHUB_REPO_URL}" "{PROJECT_DIR}"')
    os.chdir(PROJECT_DIR)

print("\nWorking directory:", os.getcwd())
print("\nContents:")
os.system("ls")

In [ ]:
# STEP 3: Install packages Colab doesn't include by default (FastAPI, Streamlit, etc).
!pip install -q fastapi uvicorn pydantic streamlit plotly nbformat nbconvert

print("Dependencies installed.")

In [ ]:
# STEP 4: Confirm the bundled dataset is present at data/raw/OnlineRetail.csv.
import os

raw_path = "data/raw/OnlineRetail.csv"

if os.path.exists(raw_path):
    size_mb = os.path.getsize(raw_path) / (1024 * 1024)
    print(f"Found dataset at {raw_path} ({size_mb:.1f} MB) — ready to go.")
else:
    print(f"No file found at {raw_path}. Check that Step 2's clone succeeded, "
          f"or upload a file manually using the cell below.")

In [ ]:
# OPTIONAL: run this only to replace the bundled CSV with the real Kaggle/UCI dataset.

# from google.colab import files
# print("Choose your real OnlineRetail.csv (or Online Retail.xlsx exported as csv)...")
# uploaded = files.upload()
# real_file = list(uploaded.keys())[0]
# import shutil
# shutil.move(real_file, "data/raw/OnlineRetail.csv")
# print("Replaced data/raw/OnlineRetail.csv with your uploaded file.")

In [ ]:
# STEP 5: Run all 6 pipeline notebooks in order (cleaning -> RFM -> PCA/tSNE -> KMeans -> DBSCAN -> export).
notebooks = [
    "01_eda.ipynb",
    "02_rfm_feature_engineering.ipynb",
    "03_pca_tsne.ipynb",
    "04_kmeans_hierarchical.ipynb",
    "05_dbscan_anomaly.ipynb",
    "06_model_export.ipynb",
]

for nb in notebooks:
    print(f"Running notebooks/{nb} ...")
    result = os.system(
        f'jupyter nbconvert --to notebook --execute --inplace "notebooks/{nb}"'
    )
    status = "OK" if result == 0 else f"FAILED (exit code {result})"
    print(f"  {nb}: {status}")

print("\nAll notebooks finished. Check output above for any FAILED steps before continuing.")

In [ ]:
# STEP 6: Verify the trained pipeline (clustering_pipeline.joblib) loaded correctly.
import joblib

bundle = joblib.load("models/clustering_pipeline.joblib")
print("Feature columns:", bundle["feature_columns"])
print("Number of clusters (K):", bundle["kmeans_model"].n_clusters)
print("Personas:")
for cid, name in sorted(bundle["persona_labels"].items()):
    print(f"  Cluster {cid}: {name}")

In [ ]:
# STEP 7a: Start Streamlit in the background inside the Colab VM (port 8501).
# Kill any previous Streamlit process from an earlier run of this cell
!pkill -f streamlit || true

# Start Streamlit in the background
os.chdir(f"{PROJECT_DIR}/dashboard")
get_ipython().system_raw(
    'streamlit run Home.py --server.port 8501 --server.headless true &>/content/logs.txt &'
)

print("Streamlit starting... waiting a few seconds for it to come up.")
import time
time.sleep(8)
print("Done. Run the next cell to get your public link.")

In [ ]:
# STEP 7b: Print the tunnel password (Colab VM's public IP) - paste this into the loca.lt page.
# Print your tunnel password (Colab's public IP)
!wget -q -O - https://loca.lt/mytunnelpassword

In [ ]:
# STEP 7c: Start the public tunnel (keep this cell running; open the printed URL in your browser).
# Start the public tunnel — this cell will keep running; that's expected.
# Copy the printed URL into your browser, then paste the IP above into the
# "Tunnel Password" field on the page that loads.
!npx localtunnel --port 8501